# Sinmate: Sinhala Tokenizer Fertility & Compression Benchmark

This notebook is the **Jupyter-server twin** of `notebooks/colab/01_colab_sinhala_tokenizer_benchmark.ipynb` (same cells, same order; environment-specific cells differ). It runs on any regular Jupyter server and needs no GPU. It evaluates how different base LLMs (Gemma-2-2B/Gemma-3, Qwen-2.5-1.5B, SinLlama) tokenize native Sinhala script, spoken colloquial phrasing (*Kathitha Bhashawa*), and Romanized Singlish.

### Why this matters:
- **Fertility < 2.5 Tokens/Word**: Model understands Sinhala natively. No embedding surgery or tokenizer retraining needed; ready for direct QLoRA fine-tuning.
- **Fertility > 4.0 Tokens/Word**: Model shatters Sinhala characters into raw UTF-8 bytes (3 bytes per letter). Requires tokenizer expansion.

In [ ]:
# 1. Install dependencies
%pip install -q transformers tabulate accelerate sentencepiece

In [ ]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: HF_TOKEN env var -> cached `huggingface-cli login` -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import get_token, login

token = os.environ.get('HF_TOKEN') or get_token() or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [ ]:
# 3. Benchmark Script Execution
import unicodedata
from transformers import AutoTokenizer
from tabulate import tabulate

MODELS_TO_TEST = {
    "Gemma-3-4B-IT": "google/gemma-3-4b-it",
    "Gemma-3-1B-IT": "google/gemma-3-1b-it"
}

TEST_SENTENCES = [
    # 1. Colloquial Spoken Sinhala (Kathitha)
    "මම අද කොළඹ යනවා, ඔයා හවසට එනවද මාව හම්බවෙන්න?",
    # 2. Formal Written Sinhala (Likhitha)
    "ශ්‍රී ලංකාවේ ආර්ථිකය සහ තොරතුරු තාක්ෂණ ක්ෂේත්‍රය පිළිබඳව විශේෂ අවධානයක් යොමු කළ යුතුය.",
    # 3. Conversational Customer Support / Technical
    "මගේ ගිණුමේ ශේෂය පරීක්ෂා කරගන්න පුළුවන් කොහොමද කියලා පැහැදිලි කරන්න.",
    # 4. Singlish + Sinhala Code-Mixing
    "Mata me API endpoint eka integrate karana widiha සිංහලෙන් explain karanna පුළුවන්ද?"
]

def evaluate_tokenizer(model_name, repo_id, sentences):
    print(f"\nEvaluating: {model_name} ({repo_id})...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    except Exception as e:
        print(f"⚠️ Could not load {repo_id}: {e}")
        return None

    total_words = 0
    total_tokens = 0
    total_chars = 0

    for idx, text in enumerate(sentences, 1):
        clean_text = unicodedata.normalize("NFC", text)
        words = clean_text.split()
        token_ids = tokenizer.encode(clean_text, add_special_tokens=False)
        decoded_pieces = [tokenizer.decode([tid]) for tid in token_ids]

        num_words = len(words)
        num_tokens = len(token_ids)
        fertility = num_tokens / num_words if num_words > 0 else 0

        total_words += num_words
        total_tokens += num_tokens
        total_chars += len(clean_text)

        if idx == 1:
            print(f"  Sentence: {clean_text}")
            print(f"  Tokens  : {decoded_pieces[:10]}...")
            print(f"  Fertility: {fertility:.2f} tokens/word")

    overall_fertility = total_tokens / total_words if total_words > 0 else 0
    chars_per_token = total_chars / total_tokens if total_tokens > 0 else 0

    return [
        model_name,
        f"{tokenizer.vocab_size:,}",
        total_words,
        total_tokens,
        f"{overall_fertility:.2f}",
        f"{chars_per_token:.2f}"
    ]

summary_rows = []
for name, repo in MODELS_TO_TEST.items():
    row = evaluate_tokenizer(name, repo, TEST_SENTENCES)
    if row:
        summary_rows.append(row)

if summary_rows:
    print("\n\n📊 FINAL TOKEN FERTILITY BENCHMARK (Lower Fertility = Better/Faster)")
    headers = ["Model", "Vocab Size", "Total Words", "Total Tokens", "Fertility (Tok/Word)", "Chars/Token"]
    print(tabulate(summary_rows, headers=headers, tablefmt="github"))